<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-13-industry-and-capstone/lesson-13.1-genai-in-healthcare/notebooks/exercises-13_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 13.1: GenAI in Healthcare

*Module 13 · Industry Applications*

> Healthcare AI has the highest stakes: a hallucination isn't an inconvenience, it's a potential patient safety risk. This lesson builds three clinical prototypes — clinical note generation, medical Q&A, and radiology reports — each wrapped in guardrails that enforce disclaimers, require physician review, and block unsafe outputs.

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-13.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Clinical Note Generator — Conversation → SOAP Note — `part1_clinical_notes.py`
2. Step 2 — Medical Q&A with Guardrails — Helpful but Safe — `part2_medical_qa.py`
3. Step 3 — Radiology Report Generator — Findings → Structured Report — `part3_radiology.py`
4. Step 4 — Complete MedicalAISystem — Unified with Audit Trail — `part4_medical_system.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Clinical Note Generator — Conversation → SOAP Note

Turn a doctor-patient conversation into a structured SOAP note in seconds.

**`part1_clinical_notes.py`** · _Block 1 of 4_

CLINICAL NOTE GENERATOR — Conversation transcript → structured SOAP note.


In [ ]:
# =============================================
# CLINICAL NOTE GENERATOR
# Conversation transcript → structured SOAP note.
# EDUCATIONAL PROTOTYPE ONLY — requires physician
# review before any clinical use.
# =============================================

import google.generativeai as genai
import os, json, re
from datetime import datetime

genai.configure(api_key=os.getenv("GOOGLE_API_KEY", ""))

class ClinicalNoteGenerator:
    """Generate SOAP notes from doctor-patient conversations."""
    
    SYSTEM_PROMPT = """You are a medical scribe AI assistant. Convert doctor-patient
conversations into structured SOAP notes.

Rules:
- Use standard medical terminology and abbreviations
- Include ONLY information explicitly stated in the conversation
- NEVER invent symptoms, findings, or diagnoses not mentioned
- NEVER recommend treatments — only document what the doctor discussed
- Flag any information that seems ambiguous with [VERIFY]
- Mark all assessment/plan items with [PHYSICIAN REVIEW REQUIRED]

Output format — return ONLY valid JSON:
{
  "subjective": {
    "chief_complaint": "...",
    "history_of_present_illness": "...",
    "past_medical_history": ["..."],
    "medications": ["..."],
    "allergies": ["..."],
    "review_of_systems": "..."
  },
  "objective": {
    "vitals": {"bp": "", "hr": "", "temp": "", "rr": "", "spo2": ""},
    "physical_exam": "...",
    "lab_results": "..."
  },
  "assessment": "[PHYSICIAN REVIEW REQUIRED] ...",
  "plan": ["[PHYSICIAN REVIEW REQUIRED] ..."],
  "flags": ["any ambiguities or missing information"]
}"""
    
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash",
            system_instruction=self.SYSTEM_PROMPT,
            generation_config={"temperature": 0.1})
    
    def generate_note(self, transcript: str, patient_id: str = "") -> dict:
        """Generate a SOAP note from a conversation transcript."""
        resp = self.model.generate_content(
            f"Convert this doctor-patient conversation into a SOAP note:\n\n{transcript}"
        )
        
        try:
            clean = re.sub(r"```json?\s*", "", resp.text).replace("```", "").strip()
            note = json.loads(clean)
        except:
            note = {"raw_text": resp.text, "parse_error": True}
        
        # Add metadata
        note["metadata"] = {
            "generated_at": datetime.now().isoformat(),
            "patient_id": patient_id,
            "status": "DRAFT — REQUIRES PHYSICIAN REVIEW",
            "model": "gemini-2.0-flash",
            "disclaimer": "AI-generated draft. Not for clinical use without physician review.",
        }
        
        return note

# ── Demo ──
generator = ClinicalNoteGenerator()

SAMPLE_TRANSCRIPT = """
Doctor: Good morning, Mrs. Sharma. What brings you in today?
Patient: I've been having this persistent headache for the past 3 days. It's mostly on the right side.
Doctor: Can you describe the pain? Is it throbbing, sharp, dull?
Patient: It's throbbing, and it gets worse when I bend over. I also feel a bit nauseous sometimes.
Doctor: Have you had any visual changes? Sensitivity to light?
Patient: Yes, bright lights bother me a lot.
Doctor: Any recent head trauma or fever?
Patient: No trauma. No fever that I've noticed.
Doctor: What medications are you currently taking?
Patient: I take Metformin for diabetes and Amlodipine for blood pressure.
Doctor: Any allergies?
Patient: Penicillin — I get a rash.
Doctor: Let me check your vitals. BP is 142/88, heart rate 76, temperature 98.4°F.
Doctor: Your neurological exam is normal. I'm going to start you on Sumatriptan 50mg as needed for the headaches. If symptoms persist beyond a week, we'll order an MRI.
Patient: Okay, thank you doctor.
"""

note = generator.generate_note(SAMPLE_TRANSCRIPT, patient_id="PT-2025-0042")
print(json.dumps(note, indent=2)[:800])


> **What just happened?** ClinicalNoteGenerator converts a doctor-patient conversation into a structured SOAP note (Subjective, Objective, Assessment, Plan). The system prompt enforces: only document what's explicitly stated (never invent symptoms), flag ambiguities with [VERIFY] , mark all assessments and plans with [PHYSICIAN REVIEW REQUIRED] . Temperature is 0.1 (minimally creative — we want accuracy, not creativity). Metadata stamps every note as "DRAFT — REQUIRES PHYSICIAN REVIEW." This saves physicians 15-20 minutes of documentation per patient encounter.


### Step 2 · Medical Q&A with Guardrails — Helpful but Safe

Answer medical questions with mandatory disclaimers, emergency detection, and scope limits.

**`part2_medical_qa.py`** · _Block 2 of 4_

MEDICAL Q&A WITH GUARDRAILS — Answer medical questions safely.


In [ ]:
# =============================================
# MEDICAL Q&A WITH GUARDRAILS
# Answer medical questions safely.
# Detect emergencies. Enforce disclaimers.
# Block diagnosis and prescription.
# =============================================

import re

class MedicalQAGuardrails:
    """Safety layer for medical Q&A."""
    
    # Emergency patterns → direct to emergency services
    EMERGENCY_PATTERNS = [
        r"chest pain", r"can'?t breathe", r"difficulty breathing",
        r"suicid", r"self.harm", r"want to die",
        r"overdose", r"poison", r"unconscious",
        r"severe bleeding", r"stroke", r"heart attack",
        r"seizure", r"anaphyla",
    ]
    
    # Questions we must NOT answer directly
    BLOCKED_PATTERNS = [
        (r"(what|which)\s+(medication|drug|medicine)\s+should I take", "prescription_request"),
        (r"(diagnose|do I have|is this)\s+(cancer|tumor|disease)", "diagnosis_request"),
        (r"(dosage|how much|how many)\s+.*(take|dose)", "dosage_request"),
        (r"stop\s+taking\s+(my|the)\s+medication", "medication_change"),
    ]
    
    MANDATORY_DISCLAIMER = (
        "\n\n⚕️ **Important:** This is general health information only, not medical advice. "
        "Please consult a qualified healthcare professional for diagnosis and treatment. "
        "If this is a medical emergency, call your local emergency number immediately."
    )
    
    def check(self, question: str) -> dict:
        """Pre-check a medical question for safety."""
        q = question.lower()
        
        # Emergency detection
        for pattern in self.EMERGENCY_PATTERNS:
            if re.search(pattern, q):
                return {
                    "action": "emergency",
                    "response": ("🚨 This sounds like it could be a medical emergency. "
                                 "Please call emergency services (112 in India, 911 in US) "
                                 "or go to your nearest emergency room immediately. "
                                 "Do not wait for an AI response."),
                }
        
        # Blocked questions
        for pattern, category in self.BLOCKED_PATTERNS:
            if re.search(pattern, q):
                return {
                    "action": "blocked",
                    "category": category,
                    "response": ("I can't provide specific medication recommendations, diagnoses, "
                                 "or dosage information. Please consult your doctor or pharmacist "
                                 "for personalized medical advice."),
                }
        
        return {"action": "allow"}

class MedicalQA:
    """Medical Q&A with safety guardrails."""
    
    SYSTEM_PROMPT = """You are a medical information assistant. Provide general health
education and information.

STRICT RULES:
- NEVER diagnose conditions
- NEVER prescribe or recommend specific medications
- NEVER provide dosage information
- NEVER tell users to stop taking medications
- ALWAYS recommend consulting a healthcare professional
- ALWAYS use phrases like "may", "could", "some people experience"
- NEVER use definitive diagnostic language like "you have" or "this is"
- Include relevant warning signs that warrant immediate medical attention
- Cite general medical knowledge, not specific studies (unless well-established)"""
    
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash",
            system_instruction=self.SYSTEM_PROMPT,
            generation_config={"temperature": 0.2})
        self.guardrails = MedicalQAGuardrails()
    
    def ask(self, question: str) -> dict:
        """Answer a medical question with safety checks."""
        
        # Pre-check
        safety = self.guardrails.check(question)
        if safety["action"] != "allow":
            return safety
        
        # Generate answer
        resp = self.model.generate_content(question)
        answer = resp.text
        
        # Post-check: ensure no diagnostic language slipped through
        dangerous_phrases = ["you have", "you are suffering from", "take this medication",
                             "I prescribe", "the diagnosis is"]
        for phrase in dangerous_phrases:
            if phrase.lower() in answer.lower():
                answer = re.sub(re.escape(phrase), "[a healthcare professional can determine if you]",
                               answer, flags=re.IGNORECASE)
        
        # Mandatory disclaimer
        answer += self.guardrails.MANDATORY_DISCLAIMER
        
        return {"action": "answered", "response": answer}

# ── Demo ──
qa = MedicalQA()

test_questions = [
    "What are common causes of headaches?",           # ✅ general info → answer
    "I'm having severe chest pain right now",          # 🚨 emergency → redirect
    "What medication should I take for my diabetes?",   # 🚫 prescription → blocked
    "What does high blood pressure mean?",              # ✅ educational → answer
]

print("Medical Q&A with guardrails:\n")
for q in test_questions:
    result = qa.ask(q)
    icons = {"answered": "✅", "emergency": "🚨", "blocked": "🚫"}
    icon = icons.get(result["action"], "❓")
    print(f"  {icon} \"{q}\"")
    print(f"     → {result['response'][:120]}...\n")


> **What just happened?** Two-layer safety: MedicalQAGuardrails (pre-check) catches emergencies ("chest pain" → redirect to 112/911), blocks prescription requests ("what medication should I take"), and blocks diagnosis requests. MedicalQA (post-check) scans the LLM's response for dangerous phrases that slipped through ("you have," "the diagnosis is") and replaces them. Every response gets a mandatory disclaimer appended. System prompt forbids diagnosis, prescription, dosage, and definitive diagnostic language. Three outcomes: answer with disclaimer (safe questions), emergency redirect (life-threatening), or block (scope violations).


### Step 3 · Radiology Report Generator — Findings → Structured Report

Turn imaging findings into a formatted radiology report draft for physician review.

**`part3_radiology.py`** · _Block 3 of 4_

RADIOLOGY REPORT GENERATOR — Imaging findings → structured report.


In [ ]:
# =============================================
# RADIOLOGY REPORT GENERATOR
# Imaging findings → structured report.
# EDUCATIONAL PROTOTYPE — requires radiologist
# review before any clinical use.
# =============================================

class RadiologyReportGenerator:
    """Generate structured radiology reports from findings."""
    
    SYSTEM_PROMPT = """You are a radiology report drafting assistant.
Generate structured radiology reports from imaging findings.

STRICT RULES:
- Use standard radiology reporting format
- Include ONLY findings provided — NEVER add findings not mentioned
- Use standard terminology (BI-RADS, Fleischner, etc. where applicable)
- ALWAYS mark the impression as [RADIOLOGIST REVIEW REQUIRED]
- Flag critical/urgent findings with [CRITICAL — NOTIFY REFERRING PHYSICIAN]
- Include comparison with prior studies ONLY if provided
- NEVER provide treatment recommendations

Output format — return ONLY valid JSON:
{
  "exam_type": "...",
  "clinical_indication": "...",
  "technique": "...",
  "findings": {
    "primary": "detailed primary findings",
    "secondary": "additional observations",
    "negative_pertinent": "relevant normal findings"
  },
  "impression": "[RADIOLOGIST REVIEW REQUIRED] numbered impression items",
  "critical_findings": [],
  "recommendation": "[RADIOLOGIST REVIEW REQUIRED] follow-up if applicable",
  "bi_rads_or_category": "if applicable"
}"""
    
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash",
            system_instruction=self.SYSTEM_PROMPT,
            generation_config={"temperature": 0.1})
    
    def generate_report(self, exam_type: str, findings: str,
                        clinical_history: str = "",
                        prior_study: str = "") -> dict:
        """Generate a radiology report from findings."""
        prompt = f"""Generate a radiology report:

Exam type: {exam_type}
Clinical history: {clinical_history}
{f'Prior study: {prior_study}' if prior_study else 'No prior studies available.'}

Findings:
{findings}"""
        
        resp = self.model.generate_content(prompt)
        
        try:
            clean = re.sub(r"```json?\s*", "", resp.text).replace("```", "").strip()
            report = json.loads(clean)
        except:
            report = {"raw_text": resp.text}
        
        report["metadata"] = {
            "status": "DRAFT — REQUIRES RADIOLOGIST REVIEW",
            "generated_at": datetime.now().isoformat(),
            "disclaimer": "AI-generated draft report. Must be reviewed and signed by a board-certified radiologist.",
        }
        
        return report

# ── Demo ──
radiology = RadiologyReportGenerator()

report = radiology.generate_report(
    exam_type="CT Chest with Contrast",
    clinical_history="55-year-old male, persistent cough for 3 months, 10 pack-year smoking history",
    findings="""
- 2.3 cm spiculated nodule in the right upper lobe
- Mediastinal lymphadenopathy, largest node 1.8 cm in station 4R
- No pleural effusion
- No bone lesions identified
- Heart size normal, no pericardial effusion
- Liver, adrenals, and visualized upper abdomen unremarkable
""",
    prior_study="CXR 2 months ago showed vague right upper lobe opacity",
)

print("Radiology Report (draft):\n")
print(json.dumps(report, indent=2)[:800])


> **What just happened?** RadiologyReportGenerator takes imaging findings and produces a structured report with: exam type, clinical indication, technique, detailed findings (primary, secondary, pertinent negatives), impression, and recommendations. The system prompt enforces: only report findings provided (never add unstated findings), use standard terminology (BI-RADS, Fleischner), mark impressions as [RADIOLOGIST REVIEW REQUIRED] , flag critical findings as [CRITICAL — NOTIFY REFERRING PHYSICIAN] . The demo shows a concerning CT chest with a spiculated nodule + lymphadenopathy — the AI flags this as critical.


### Step 4 · Complete MedicalAISystem — Unified with Audit Trail

All tools with physician review gates, audit logging, and regulatory compliance.

**`part4_medical_system.py`** · _Block 4 of 4_

COMPLETE MEDICAL AI SYSTEM — Unified tools with audit trail, physician


In [ ]:
# =============================================
# COMPLETE MEDICAL AI SYSTEM
# Unified tools with audit trail, physician
# review gates, and compliance logging.
# =============================================

class MedicalAISystem:
    """Medical AI tools with full audit trail and safety gates."""
    
    def __init__(self):
        self.notes = ClinicalNoteGenerator()
        self.qa = MedicalQA()
        self.radiology = RadiologyReportGenerator()
        self.audit_log: list[dict] = []
    
    def _log(self, tool: str, input_summary: str, output_summary: str,
            status: str, physician_id: str = ""):
        """Immutable audit log entry for every AI action."""
        self.audit_log.append({
            "timestamp": datetime.now().isoformat(),
            "tool": tool,
            "input_summary": input_summary[:100],
            "output_summary": output_summary[:100],
            "status": status,
            "physician_review": "pending" if not physician_id else f"reviewed_by:{physician_id}",
        })
    
    def generate_clinical_note(self, transcript: str, patient_id: str) -> dict:
        note = self.notes.generate_note(transcript, patient_id)
        self._log("clinical_note", f"Patient:{patient_id}",
                  note.get("assessment", "")[:100], "draft_generated")
        return note
    
    def answer_medical_question(self, question: str) -> dict:
        result = self.qa.ask(question)
        self._log("medical_qa", question, result.get("response", "")[:100], result["action"])
        return result
    
    def generate_radiology_report(self, exam_type: str, findings: str,
                                  clinical_history: str = "") -> dict:
        report = self.radiology.generate_report(exam_type, findings, clinical_history)
        self._log("radiology_report", exam_type, 
                  report.get("impression", "")[:100], "draft_generated")
        return report
    
    def physician_review(self, log_index: int, physician_id: str,
                         approved: bool, notes: str = ""):
        """Record physician review of an AI-generated document."""
        if 0 <= log_index < len(self.audit_log):
            self.audit_log[log_index]["physician_review"] = {
                "physician_id": physician_id,
                "approved": approved,
                "review_notes": notes,
                "reviewed_at": datetime.now().isoformat(),
            }
    
    def compliance_report(self):
        """Generate a compliance summary."""
        total = len(self.audit_log)
        reviewed = sum(1 for e in self.audit_log if isinstance(e.get("physician_review"), dict))
        pending = total - reviewed
        
        print(f"\n  📋 Medical AI Compliance Report")
        print(f"  {'─'*45}")
        print(f"  Total AI actions:       {total}")
        print(f"  Physician reviewed:     {reviewed}")
        print(f"  Pending review:         {pending}")
        print(f"  Review rate:            {reviewed/max(total,1):.0%}")
        
        if pending > 0:
            print(f"  ⚠️ {pending} documents awaiting physician review")
        
        for i, e in enumerate(self.audit_log):
            status_icon = "✅" if isinstance(e.get("physician_review"), dict) else "⏳"
            print(f"    {status_icon} [{e['tool']:17s}] {e['status']:18s} | {e['input_summary'][:40]}")

# ── Demo ──
system = MedicalAISystem()

print("MedicalAISystem demo:\n")

# Generate a clinical note
system.generate_clinical_note(SAMPLE_TRANSCRIPT, "PT-2025-0042")
print("  ✅ Clinical note generated (pending review)")

# Answer a medical question
system.answer_medical_question("What are common causes of headaches?")
print("  ✅ Medical Q&A answered")

# Emergency detection
system.answer_medical_question("I'm having severe chest pain")
print("  🚨 Emergency detected and redirected")

# Simulate physician review
system.physician_review(0, "DR-PATEL-1234", approved=True, notes="Reviewed, minor edits to HPI.")
print("  ✅ Clinical note reviewed by Dr. Patel")

# Compliance report
system.compliance_report()


> **What just happened?** MedicalAISystem unifies all three tools with: (1) Audit logging — every AI action logged with timestamp, tool, input/output summary, and review status. (2) Physician review gates — every generated document starts as "pending review." The physician_review() method records who reviewed it, when, and whether they approved. (3) Compliance reporting — shows total actions, review rate, and documents still awaiting review. In production: no AI-generated clinical document is released to the patient record until a physician signs off. The audit log provides the regulatory paper trail.


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
